<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/ecomarket-solution/blob/main/notebooks/EcoMarket_AI_Solution.ipynb)


# Maestría en Inteligencia Artificial  
## IA Generativa
---
# EcoMarket AI Support — Taller Práctico #1
## Optimización de Atención al Cliente con IA Generativa

**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

**Modelo:** `llama-3.3-70b-versatile` via [Groq API](https://console.groq.com)


In [1]:
import warnings
from importlib import metadata

warnings.filterwarnings('ignore')

installed_packages = {dist.metadata['Name'].lower() for dist in metadata.distributions() if dist.metadata.get('Name')}
IN_COLAB = 'google-colab' in installed_packages

In [2]:
# CELDA 1 — Instalación de dependencias
!pip install -q langchain langchain-groq
!pip install -q requests colorama
print('Dependencias instaladas')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.9 MB/s eta 0:00:00
✅ Dependencias instaladas


Cargar data

In [5]:
!test '{IN_COLAB}' = 'True' && mkdir -p data && [ ! -f data/orders_database.txt ] && wget -q -O data/orders_database.txt https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/data/orders_database.txt

!test '{IN_COLAB}' = 'True' && mkdir -p data && [ ! -f data/return_policy.txt ] && wget -q -O data/return_policy.txt https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/data/return_policy.txt

Cargar prompts

In [9]:
!test '{IN_COLAB}' = 'True' && mkdir -p prompts && [ ! -f prompts/system_prompts.txt ] && wget -q -O prompts/system_prompts.txt https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/prompts/system_prompts.txt

!test '{IN_COLAB}' = 'True' && mkdir -p prompts && [ ! -f prompts/order_status_prompt.txt ] && wget -q -O prompts/order_status_prompt.txt https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/prompts/order_status_prompt.txt

!test '{IN_COLAB}' = 'True' && mkdir -p prompts && [ ! -f prompts/return_policy_prompt.txt ] && wget -q -O prompts/return_policy_prompt.txt https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/prompts/return_policy_prompt.txt

In [19]:
# ============================================================
# CONFIGURACIÓN SEGURA GROQ (COLAB + SECRETS)
# ============================================================

import os
from google.colab import userdata

def get_api_key():
    """
    Obtiene la API Key de forma segura desde Colab Secrets.
    Lanza error si no existe.
    """
    api_key = userdata.get("GROQ_API_KEY")

    if not api_key:
        raise ValueError("❌ No se encontró GROQ_API_KEY en Colab Secrets")

    if not api_key.startswith("gsk_"):
        raise ValueError("❌ La API Key no tiene formato válido")

    return api_key

# ========================
# VARIABLES DE CONFIG
# ========================

GROQ_API_KEY = get_api_key()

CONFIG = {
    "model": "llama-3.3-70b-versatile",
    "api_url": "https://api.groq.com/openai/v1/chat/completions",
    "max_tokens": 1024,
    "temperature": 0.3,
    "timeout": 30,  # buena práctica
}

# ========================
# HEADERS LISTOS
# ========================

HEADERS = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json"
}

print(f"✅ API Key cargada correctamente ({GROQ_API_KEY[:6]}****)")
print(f"🤖 Modelo configurado: {CONFIG['model']}")

✅ API Key cargada correctamente (gsk_Gw****)
🤖 Modelo configurado: llama-3.3-70b-versatile


In [20]:
# Carga de datos de ordenes y politicas

with open("data/orders_database.txt", "r", encoding="utf-8") as f:
    ORDERS_DATABASE = f.read()

with open("data/return_policy.txt", "r", encoding="utf-8") as f:
    RETURN_POLICY = f.read()

print("Datos cargados")

Datos cargados


In [21]:
# System Prompts

with open("prompts/system_prompts.txt", "r", encoding="utf-8") as f:
    ECOBOT_BASE = f.read()

with open("prompts/order_status_prompt.txt", "r", encoding="utf-8") as f:
    ORDER_SYSTEM = ECOBOT_BASE + "\n\n" + f.read()

with open("prompts/return_policy_prompt.txt", "r", encoding="utf-8") as f:
    RETURN_SYSTEM = ECOBOT_BASE + "\n\n" + f.read()

print("System prompts cargados")

System prompts cargados


In [22]:
# ============================================================
# CELDA 6 — Cliente LLM (adaptado a nueva configuración)
# ============================================================

import requests

def call_llm(system_prompt: str, user_message: str) -> str:
    """Llama a la API de Groq y retorna la respuesta generada."""

    # ✅ Ya no necesitas validar contra string hardcodeado
    if not GROQ_API_KEY:
        return "⚠️ No se encontró la API Key. Revisa Colab Secrets."

    payload = {
        "model": CONFIG["model"],
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ],
        "max_tokens": CONFIG["max_tokens"],
        "temperature": CONFIG["temperature"]
    }

    try:
        r = requests.post(
            CONFIG["api_url"],
            headers=HEADERS,
            json=payload,
            timeout=CONFIG["timeout"]
        )

        # ========================
        # MANEJO DE ERRORES
        # ========================
        if r.status_code == 401:
            return "❌ API Key inválida. Verifica en console.groq.com"

        if r.status_code != 200:
            return f"❌ Error HTTP {r.status_code}: {r.text[:200]}"

        data = r.json()

        # ========================
        # VALIDACIÓN RESPUESTA
        # ========================
        if "choices" not in data or len(data["choices"]) == 0:
            return "❌ Respuesta inválida del modelo"

        return data["choices"][0]["message"]["content"].strip()

    except requests.exceptions.Timeout:
        return "⏱️ Timeout: el modelo tardó demasiado en responder"

    except requests.exceptions.ConnectionError:
        return "🌐 Error de conexión: revisa tu internet"

    except Exception as e:
        return f"❌ Error: {str(e)}"


print("✅ Cliente LLM listo (adaptado a CONFIG)")

✅ Cliente LLM listo (adaptado a CONFIG)


In [23]:
# CELDA 7 — Funciones de Prompt Engineering

def query_order_status(tracking_number: str) -> str:
    """
    Consulta el estado de un pedido.
    Técnicas: RAG simulado, instrucciones paso a paso,
    manejo de casos extremos, restricción anti-alucinación.
    """
    user_prompt = f"""--- BASE DE DATOS DE PEDIDOS (usa ÚNICAMENTE esta información) ---
{ORDERS_DATABASE}
--- FIN DEL CONTEXTO ---

El cliente pregunta por su pedido: {tracking_number}

INSTRUCCIONES:
PASO 1 — Busca '{tracking_number}' en el contexto (mayúsculas/minúsculas, con y sin guión).

PASO 2a — Si el pedido EXISTE:
  - Estado actual con descripción clara
  - Fecha estimada de entrega o confirmación si ya fue entregado
  - URL de tracking si está disponible
  - Si RETRASADO: disculpa sincera primero, luego nueva fecha y razón
  - Si CANCELADO: estado del reembolso y plazos exactos
  - Si PENDIENTE DE PAGO: urgencia amable + enlace de pago
  - Próximo paso que el cliente debe esperar

PASO 2b — Si el pedido NO APARECE en el contexto:
  - Informa amablemente que no puedes encontrar ese número
  - Sugiere verificar el número (puede haber un error tipográfico)
  - Ofrece contacto: soporte@ecomarket.com o 900-ECO-MKT
  - NUNCA inventes datos de un pedido que no aparece en el contexto

FORMATO: Saludo personalizado + información + oferta de ayuda adicional."""
    return call_llm(ORDER_SYSTEM, user_prompt)


def request_return(product_name: str, reason: str, days: int,
                   opened: bool = False, first_return: bool = False) -> str:
    """
    Procesa una solicitud de devolución.
    Técnicas: árbol de decisión explícito, empatía antes de negativa,
    excepciones documentadas, nunca un 'no' sin alternativa.
    """
    opened_txt = "Sí" if opened else "No especificado"
    first_txt  = "Sí (posible excepción hasta 45 días)" if first_return else "No"

    user_prompt = f"""--- POLÍTICA DE DEVOLUCIONES DE ECOMARKET ---
{RETURN_POLICY}
--- FIN DE LA POLÍTICA ---

SOLICITUD DEL CLIENTE:
  Producto: {product_name}
  Motivo: {reason}
  Días desde la entrega: {days}
  ¿Producto abierto?: {opened_txt}
  ¿Primera devolución del cliente?: {first_txt}

ÁRBOL DE DECISIÓN (sigue en orden):

PASO 1 — Excepción universal:
  ¿El motivo menciona daño en envío, defecto o producto incorrecto?
  -> SÍ: APRUEBA siempre. Cubre el envío. Ofrece reembolso O reenvío. FIN.
  -> NO: continúa al Paso 2.

PASO 2 — Categoría del producto:
  ¿Es alimento, bebida, planta, suplemento? -> NO posible (sanitario). -> Paso 5.
  ¿Es higiene personal Y fue abierto? -> NO posible. -> Paso 5.
  ¿Es personalizado o digital? -> NO posible. -> Paso 5.
  Ninguna aplica -> puede devolverse. -> Paso 3.

PASO 3 — Plazo de 30 días:
  ¿Más de 30 días Y es primera devolución? -> Escala a agente humano, excepción posible.
  ¿Más de 30 días sin excepción? -> NO posible. -> Paso 5.
  ¿Dentro de 30 días? -> Devolución aprobada. -> Paso 4.

PASO 4 — RESPUESTA AFIRMATIVA:
  Confirma, explica el proceso en 3 pasos numerados, plazo del reembolso,
  ofrece crédito de tienda (+5% de bonificación).

PASO 5 — NEGATIVA EMPÁTICA:
  1. Empatía genuina primero (reconoce la frustración)
  2. Razón de forma simple y humana (no párrafo legal)
  3. Al menos UNA alternativa concreta
  4. NUNCA termines con un 'no' definitivo y frío sin ofrecer algo"""
    return call_llm(RETURN_SYSTEM, user_prompt)


def print_response(title: str, response: str):
    print(f"\n{'='*65}")
    print(f"📋 {title}")
    print('='*65)
    print(f"\n🤖 EcoBot:\n")
    print(response)
    print()

print("✅ Funciones de prompt engineering listas — ¡sistema listo para usar!")

✅ Funciones de prompt engineering listas — ¡sistema listo para usar!


---
## 📦 Ejercicio 1 — Consultas de Estado de Pedido
Demuestra el manejo de 5 estados distintos y la protección anti-alucinación.

In [24]:
# Pedido EN TRÁNSITO — información de entrega y tracking
response = query_order_status("ECO-12345")
print_response("ECO-12345 — Pedido EN TRÁNSITO — información de entrega y tracking", response)


📋 ECO-12345 — Pedido EN TRÁNSITO — información de entrega y tracking

🤖 EcoBot:

Hola María García,

Me alegra poder ayudarte con tu consulta sobre el estado de tu pedido ECO-12345. Después de verificar en nuestra base de datos, puedo informarte que tu pedido está actualmente **EN TRÁNSITO**. Esto significa que tu kit de bambú reutilizable (3 piezas) ya ha sido enviado y está en camino hacia ti.

La fecha estimada de entrega para tu pedido es el **2024-07-05**. Puedes seguir el progreso de tu pedido a través del enlace de seguimiento proporcionado por DHL Express: https://track.dhl.com/ECO12345. Esto te permitirá tener una visión clara de dónde se encuentra tu paquete en cada momento.

Quiero asegurarme de que estés satisfecha con el proceso y que no tengas ninguna inquietud. Si necesitas ayuda adicional o tienes alguna pregunta sobre tu pedido, no dudes en hacérmelo saber. Estoy aquí para ayudarte.

¿Hay algo más en lo que pueda asistirte? ¿Necesitas ayuda con algo relacionado con tu

In [25]:
# Pedido RETRASADO — debe incluir disculpa empática primero
response = query_order_status("ECO-12346")
print_response("ECO-12346 — Pedido RETRASADO — debe incluir disculpa empática primero", response)


📋 ECO-12346 — Pedido RETRASADO — debe incluir disculpa empática primero

🤖 EcoBot:

Hola Carlos,

Entiendo que estás buscando información sobre el estado de tu pedido ECO-12346. Lamentablemente, tu pedido ha sido retrasado. Quiero empezar disculpándome sinceramente por las molestias que esto te pueda haber causado. Sabemos lo importante que es recibir tus productos a tiempo, y nos esforzamos por mejorar constantemente nuestro servicio.

La nueva fecha estimada de entrega para tu pedido es el 2024-07-08. El retraso se debe a una huelga de transporte regional en Cataluña, que ha afectado a varios de nuestros envíos. Puedes seguir el progreso de tu pedido a través de la URL de seguimiento: https://track.correos.es/ECO12346.

Queremos asegurarnos de que estás satisfecho con tu experiencia en EcoMarket. Si tienes alguna otra pregunta o necesitas ayuda adicional, no dudes en hacérmelo saber. Estoy aquí para ayudarte. También puedes contactar con nuestro equipo de soporte a través del correo

In [26]:
# Pedido CANCELADO — debe informar el estado del reembolso
response = query_order_status("ECO-12349")
print_response("ECO-12349 — Pedido CANCELADO — debe informar el estado del reembolso", response)


📋 ECO-12349 — Pedido CANCELADO — debe informar el estado del reembolso

🤖 EcoBot:

Hola Sofia,

Me dirijo a ti porque he encontrado un pedido a tu nombre con el número ECO-12349. Desafortunadamente, tu pedido ha sido cancelado según nuestra base de datos. La cancelación se realizó el 2024-07-01 a petición tuya.

En cuanto al estado del reembolso, te informo que ya ha sido procesado el 2024-07-02. Debes recibir el reembolso en un plazo de 5 a 7 días hábiles en la tarjeta original terminada en ****4521.

Si tienes alguna pregunta adicional o necesitas asistencia con algo más, no dudes en preguntar. Estoy aquí para ayudarte. También puedes contactar con nuestro equipo de soporte a través del correo electrónico soporte@ecomarket.com o llamando al 900-ECO-MKT, si lo prefieres.

¿Hay algo más en lo que pueda ayudarte hoy?



In [27]:
# Pedido PENDIENTE DE PAGO — urgencia amable
response = query_order_status("ECO-12351")
print_response("ECO-12351 — Pedido PENDIENTE DE PAGO — urgencia amable", response)


📋 ECO-12351 — Pedido PENDIENTE DE PAGO — urgencia amable

🤖 EcoBot:

Hola Isabella,

Me alegra poder ayudarte con tu consulta sobre el pedido ECO-12351. Después de verificar en nuestra base de datos, encontré que tu pedido se encuentra en estado "PENDIENTE DE PAGO". Esto significa que, aunque has realizado el pedido, el pago no ha sido confirmado aún.

Es importante que completes el pago lo antes posible para que podamos procesar tu pedido sin retrasos. Puedes hacer clic en el siguiente enlace para completar el pago: https://ecomarket.com/pago/ECO12351. Ten en cuenta que, si no se completa el pago en las próximas 24 horas, el pedido se cancelará automáticamente.

Si tienes alguna pregunta o necesitas ayuda con el proceso de pago, no dudes en hacérmelo saber. Estoy aquí para ayudarte. También puedes contactar con nuestro equipo de soporte en soporte@ecomarket.com o llamando al 900-ECO-MKT si prefieres.

¿Hay algo más en lo que pueda ayudarte? Estoy a tu disposición para resolver cualqu

In [28]:
# Número INEXISTENTE — prueba anti-alucinación
response = query_order_status("ECO-99999")
print_response("ECO-99999 — Número INEXISTENTE — prueba anti-alucinación", response)


📋 ECO-99999 — Número INEXISTENTE — prueba anti-alucinación

🤖 EcoBot:

Hola, lamentablemente no puedo encontrar el pedido con el número ECO-99999 en nuestra base de datos. Es posible que haya un error tipográfico en el número de pedido o que no esté registrado en nuestro sistema.

Te sugiero verificar el número de pedido para asegurarte de que esté correcto. Si sigues teniendo problemas para encontrar tu pedido, no dudes en contactarnos a través de nuestro correo electrónico soporte@ecomarket.com o llamando al 900-ECO-MKT. Estamos aquí para ayudarte.

¿Hay algo más en lo que pueda asistirte mientras tanto? Estoy aquí para ayudarte con cualquier pregunta o inquietud que tengas sobre nuestros productos o servicios.



---
## 🔄 Ejercicio 2 — Solicitudes de Devolución
El desafío: responder con empatía tanto el «sí» como el «no»,
y nunca dar una negativa sin ofrecer una alternativa.

In [30]:
# Devolución APROBADA — producto de hogar dentro del plazo
response = request_return(product_name="Botella de acero inoxidable 750ml", reason="No me gusta el diseño", days=12, opened=False)
print_response("Devolución APROBADA — producto de hogar dentro del plazo", response)


📋 Devolución APROBADA — producto de hogar dentro del plazo

🤖 EcoBot:

Hola, gracias por contactar con EcoMarket. Entiendo que deseas devolver la botella de acero inoxidable de 750ml porque no te gusta el diseño. Me parece una razón completamente válida para querer cambiar de opinión.

Después de revisar nuestra política de devoluciones, me alegra informarte que la devolución es posible. La botella de acero inoxidable se encuentra dentro de las categorías de productos que podemos aceptar para devolución, siempre y cuando esté en su embalaje original y no haya sido utilizada.

Aquí te explico el proceso de devolución en 3 pasos sencillos:

1. **Solicita la devolución**: Accede a nuestra página web en ecomarket.com/devoluciones, introduce tu número de pedido y el motivo de la devolución.
2. **Recibe la etiqueta de envío**: En un plazo de 24 horas hábiles, recibirás una etiqueta de envío prepagada por correo electrónico. Esto te permitirá enviar el producto de vuelta a nosotros sin costo

In [31]:
# Devolución RECHAZADA — higiene personal abierto
response = request_return(product_name="Jabón orgánico artesanal", reason="No me gusta el olor", days=5, opened=True)
print_response("Devolución RECHAZADA — higiene personal abierto", response)


📋 Devolución RECHAZADA — higiene personal abierto

🤖 EcoBot:

Lo siento mucho, entiendo perfectamente tu situación y la frustración que te causa no gustarte el olor del jabón orgánico artesanal. Me imagino que esperabas algo diferente y es comprensible que quieras devolverlo.

La razón por la que no podemos aceptar la devolución en este caso es porque el producto pertenece a la categoría de higiene personal y ha sido abierto. Nuestra política de devoluciones establece que los productos de higiene personal abiertos no pueden ser devueltos por razones de higiene y seguridad.

Sin embargo, no quiero dejar la situación así. Te ofrezco una alternativa: podrías considerar donar el jabón a alguien que pueda apreciarlo o encontrarle un uso alternativo. Además, si estás interesado en encontrar un jabón que se adapte mejor a tus preferencias, puedo ofrecerte un 10% de descuento en tu próxima compra de productos de higiene personal en EcoMarket.

Si prefieres, también puedo proporcionarte inform

In [32]:
# Devolución RECHAZADA — producto perecedero (alimento)
response = request_return(product_name="Mix de frutos secos orgánicos", reason="Cambié de opinión", days=3, opened=False)
print_response("Devolución RECHAZADA — producto perecedero (alimento)", response)


📋 Devolución RECHAZADA — producto perecedero (alimento)

🤖 EcoBot:

Hola, gracias por contactar con EcoMarket. Entiendo que deseas devolver un producto, específicamente un mix de frutos secos orgánicos, debido a un cambio de opinión. Me alegra que hayas considerado nuestra política de devoluciones.

Desafortunadamente, nuestros productos alimenticios, incluyendo los frutos secos orgánicos, no pueden ser devueltos una vez enviados, debido a razones de seguridad alimentaria e higiene. Esto es para garantizar la calidad y la seguridad de nuestros productos para todos nuestros clientes.

Entiendo que esto puede ser frustrante, especialmente si has cambiado de opinión. Quiero asegurarte que estamos comprometidos con la satisfacción de nuestros clientes y queremos ayudar en lo que podamos. Aunque no podemos ofrecer un reembolso o cambio en este caso, te invitamos a considerar otras opciones de productos que podrían interesarte. Nuestro catálogo incluye una amplia variedad de artículos soste

In [33]:
# Devolución APROBADA — daño en tránsito (excepción universal)
response = request_return(product_name="Cepillo de dientes de bambú", reason="El paquete llegó aplastado y roto", days=4, opened=True)
print_response("Devolución APROBADA — daño en tránsito (excepción universal)", response)


📋 Devolución APROBADA — daño en tránsito (excepción universal)

🤖 EcoBot:

Entiendo perfectamente tu situación, y me parece muy frustrante que el paquete haya llegado dañado. Me imagino cómo te sentirías al recibir un producto en mal estado.

En este caso, como el motivo de la devolución es el daño en el envío, podemos proceder con la devolución del cepillo de dientes de bambú. A continuación, te explicaré el proceso:

1. **Solicitar la devolución**: Ya has iniciado este proceso, pero para formalizarlo, te pediré que accedas a [ecomarket.com/devoluciones](http://ecomarket.com/devoluciones) y completes la solicitud con tu número de pedido y motivo de devolución.
2. **Recibir la etiqueta de envío**: En un plazo de 24 horas hábiles, recibirás una etiqueta de envío prepagada por correo electrónico. Esto te permitirá enviar el producto de vuelta a nosotros sin costo adicional para ti.
3. **Enviar el producto**: Por favor, empaca el producto en su embalaje original si es posible y deposítal

In [34]:
# Devolución FUERA DE PLAZO — primera vez, escala a agente
response = request_return(product_name="Set de cubiertos de bambú", reason="Me lo regalaron y no lo necesito", days=38, opened=False, first_return=True)
print_response("Devolución FUERA DE PLAZO — primera vez, escala a agente", response)


📋 Devolución FUERA DE PLAZO — primera vez, escala a agente

🤖 EcoBot:

Hola, gracias por contactar con EcoMarket. Entiendo que has recibido un set de cubiertos de bambú como regalo y no lo necesitas. Me parece una situación un poco incómoda, ya que no es algo que hayas elegido tú mismo.

Desafortunadamente, el plazo de devolución de 30 días naturales desde la entrega ha expirado, ya que han pasado 38 días. Sin embargo, como es tu primera devolución con nosotros, podemos considerar una excepción. Nuestra política permite que los gestores aprueben devoluciones fuera de plazo hasta 45 días para clientes nuevos, siempre y cuando el producto no sea de las categorías prohibidas.

En este caso, el set de cubiertos de bambú es un producto que podemos aceptar para devolución. Por lo tanto, te propongo que sigamos adelante con el proceso de devolución.

Aquí te explico los pasos a seguir:

1. **Solicitar la devolución**: Ya has iniciado este proceso al contactar con nosotros. Ahora, te pediré q

---
## 🧪 Suite de Tests Automáticos
Verifica empatía en negativas, anti-alucinación y presencia de alternativas.

In [35]:
def run_tests():
    tests = [
        {
            "id": "T-ORD-01", "desc": "Pedido en tránsito reconocido",
            "fn": lambda: query_order_status("ECO-12345"),
            "checks": [(lambda r: "tránsit" in r.lower() or "camino" in r.lower(),
                         "Debe mencionar estado en tránsito")]
        },
        {
            "id": "T-ORD-02", "desc": "Pedido retrasado incluye disculpa",
            "fn": lambda: query_order_status("ECO-12346"),
            "checks": [(lambda r: any(w in r.lower() for w in ["disculp","lament","sentimos"]),
                         "Debe incluir disculpa empática")]
        },
        {
            "id": "T-ORD-03", "desc": "Anti-alucinación: pedido inexistente",
            "fn": lambda: query_order_status("ECO-99999"),
            "checks": [
                (lambda r: not any(w in r.lower() for w in ["en tránsito","entregado","retrasado"]),
                 "NO debe inventar estado para pedido inexistente"),
                (lambda r: any(w in r.lower() for w in ["encontr","verific","contact"]),
                 "Debe sugerir alternativa de contacto")
            ]
        },
        {
            "id": "T-RET-01", "desc": "Devolución de hogar aprobada",
            "fn": lambda: request_return("Botella acero", "no me gusta", 10, False),
            "checks": [(lambda r: any(w in r.lower() for w in ["devoluci","proceso","ecomarket"]),
                         "Debe explicar el proceso de devolución")]
        },
        {
            "id": "T-RET-02", "desc": "Negativa de higiene incluye empatía + alternativa",
            "fn": lambda: request_return("Jabón orgánico", "no me gusta", 5, True),
            "checks": [
                (lambda r: any(w in r.lower() for w in ["entend","comprend","lament","sentim"]),
                 "La negativa debe incluir empatía genuina"),
                (lambda r: any(w in r.lower() for w in ["alternativ","agente","crédito","contact"]),
                 "La negativa debe ofrecer una alternativa")
            ]
        },
        {
            "id": "T-RET-03", "desc": "Daño en tránsito siempre aprobado",
            "fn": lambda: request_return("Cepillo bambú", "llegó roto y aplastado", 4, True),
            "checks": [(lambda r: not any(w in r.lower() for w in ["no podemos","no es posible","no acepta"]),
                         "Daño en tránsito no debe rechazarse nunca")]
        },
    ]

    print("\n" + "="*65)
    print("  🧪 SUITE DE TESTS — EcoMarket AI Support")
    print("="*65)
    results = []
    for test in tests:
        print(f"\n  ▶ {test['id']}: {test['desc']}...", end=" ", flush=True)
        try:
            resp = test["fn"]()
            fails = [msg for chk, msg in test["checks"] if not chk(resp)]
            if fails:
                print("❌ FALLÓ")
                for f in fails:
                    print(f"     -> {f}")
                results.append(False)
            else:
                print("✅ PASÓ")
                results.append(True)
        except Exception as e:
            print(f"💥 ERROR: {e}")
            results.append(False)
    passed = sum(results)
    print(f"\n{'─'*65}")
    print(f"  Resultado: {passed}/{len(results)} tests pasaron")
    if passed == len(results):
        print("  🎉 ¡Todos los tests pasaron!")
    print()

run_tests()


  🧪 SUITE DE TESTS — EcoMarket AI Support

  ▶ T-ORD-01: Pedido en tránsito reconocido... ✅ PASÓ

  ▶ T-ORD-02: Pedido retrasado incluye disculpa... ✅ PASÓ

  ▶ T-ORD-03: Anti-alucinación: pedido inexistente... ✅ PASÓ

  ▶ T-RET-01: Devolución de hogar aprobada... ✅ PASÓ

  ▶ T-RET-02: Negativa de higiene incluye empatía + alternativa... ❌ FALLÓ
     -> La negativa debe incluir empatía genuina
     -> La negativa debe ofrecer una alternativa

  ▶ T-RET-03: Daño en tránsito siempre aprobado... ✅ PASÓ

─────────────────────────────────────────────────────────────────
  Resultado: 5/6 tests pasaron



---
## 🎮 Modo Interactivo
Cambia los valores de las variables y ejecuta las celdas para probar con tus propios datos.

In [36]:
# CONSULTA DE PEDIDO PERSONALIZADA
# Pedidos disponibles: ECO-12345 al ECO-12354, o prueba uno inexistente
MI_PEDIDO = "ECO-12353"  # <- cambia aquí

response = query_order_status(MI_PEDIDO)
print_response(f"Consulta personalizada: {MI_PEDIDO}", response)


📋 Consulta personalizada: ECO-12353

🤖 EcoBot:

Hola Valentina,

Me alegra poder ayudarte con tu consulta sobre el pedido ECO-12353. Después de verificar en nuestra base de datos, encontré que tu pedido está en el estado "LISTO PARA RECOGIDA EN TIENDA". Esto significa que tu producto, el contenedor de vidrio hermético (set x5), ya está disponible para que lo recojas en nuestra tienda física ubicada en C/ Gran Vía 45, Local 2, en Madrid.

La recogida está disponible de lunes a sábado, desde las 10:00 hasta las 20:00 horas, y debes presentar tu DNI o el número de pedido para poder recoger tu compra. Ten en cuenta que el producto estará disponible para recoger hasta el 10 de julio de 2024. Después de esta fecha, si no has recogido tu pedido, se procederá a devolver el importe.

Si tienes alguna pregunta adicional o necesitas ayuda con la recogida, no dudes en hacérmelo saber. Estoy aquí para ayudarte. Además, si necesitas asistencia con cualquier otro tema relacionado con EcoMarket, pued

In [37]:
# SOLICITUD DE DEVOLUCIÓN PERSONALIZADA
MI_PRODUCTO  = "Velas aromáticas de cera de soja"  # <- cambia aquí
MI_MOTIVO    = "No me gusta el aroma"               # <- cambia aquí
DIAS         = 8                                    # <- días desde la entrega
FUE_ABIERTO  = False                                # <- True o False
PRIMERA_VEZ  = False                                # <- True si es primera devolución

response = request_return(MI_PRODUCTO, MI_MOTIVO, DIAS, FUE_ABIERTO, PRIMERA_VEZ)
print_response(f"Devolución personalizada: {MI_PRODUCTO}", response)


📋 Devolución personalizada: Velas aromáticas de cera de soja

🤖 EcoBot:

Hola, gracias por contactar con EcoMarket. Entiendo que deseas devolver las velas aromáticas de cera de soja porque no te gusta el aroma. Me parece una razón perfectamente válida para querer devolver un producto.

Después de revisar nuestra política de devoluciones, puedo confirmarte que las velas aromáticas de cera de soja sí pueden devolverse, ya que no pertenecen a ninguna de las categorías de productos que no pueden ser devueltos, como alimentos o productos de higiene personal abiertos.

Aquí te explico el proceso de devolución en 3 pasos sencillos:

1. **Solicita la devolución**: Accede a nuestra página web en ecomarket.com/devoluciones, introduce tu número de pedido y el motivo de la devolución. Ten en cuenta que debes hacerlo dentro de los 30 días naturales desde la entrega.
2. **Recibe la etiqueta de envío**: En un plazo de 24 horas hábiles, recibirás una etiqueta de envío prepagada por correo electrónico